# 04 — Data Relationships

**Day 1, Step 4.** Decide, and write down, how the five tables connect.

In [1]:
import sys, warnings
sys.path.insert(0, "../src"); sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# The lab imports from the factory. Nothing below reimplements pipeline logic.
from hrai.utils.config import get, raw_path, seed
from hrai.utils.io import load_raw, load_processed
from hrai.utils.logger import setup_logging
setup_logging(fmt="human")
print(f"seed={seed()}  |  datasets: {sorted(get('datasets'))}")

seed=42  |  datasets: ['employee_attrition', 'essential_skills', 'hr_performance_engagement', 'occupation_data', 'software_skills']


## The corrected topology

```
  POPULATION A                          POPULATION B
  employee_attrition (1,470)            hr_performance_engagement (3,000)
  label: Attrition        16.1%         label: Voluntarily Terminated  10.8%
        |                                            |
        |  X----- NO EMPLOYEE-LEVEL JOIN (F1) -----X |
        |                                            |
        | JobRole (9)                     Title (31) |
        v                                            v
        +------>  ROLE -> O*NET SOC CROSSWALK  <-----+
                  semantic + human-reviewed, 40 roles
                              |
                              v
                    occupation_master (1,016)
                      |                    |
                      v                    v
          essential_skills          software_skills
          TIER 1 foundational       TIER 2 technical
```

The two populations are bridged by the **role/skill ontology**, not by employee
identity. This is the honest bridge, and it also means the skills layer covers
4,470 employees instead of 1,470.

In [2]:
occ = load_processed("occupation_master")
ess = load_processed("essential_skills_processed")
sw  = load_processed("software_skills_processed")

codes = set(occ["soc_code"])
print("essential_skills subset of occupations:", set(ess["soc_code"]) <= codes)
print("software_skills  subset of occupations:", set(sw["soc_code"]) <= codes)
print(f"\noccupations: {len(occ):,}  |  foundational rows: {len(ess):,}  "
      f"|  technical rows: {len(sw):,}")

essential_skills subset of occupations: True
software_skills  subset of occupations: True

occupations: 1,016  |  foundational rows: 9,079  |  technical rows: 31,706


## Population B has its own label

This is what turns finding F1 from a pure constraint into an opportunity: a
model trained on Population A can be **externally validated** against Population
B's own outcomes rather than merely assumed to generalise. See notebook 17.

In [3]:
a = load_processed("employee_attrition_processed")
b = load_processed("engagement_processed")
print(f"Population A  n={len(a):,}  label=attrition_flag      rate={a['attrition_flag'].mean():.1%}")
print(f"Population B  n={len(b):,}  label=is_voluntary_exit   rate={b['is_voluntary_exit'].mean():.1%}")

Population A  n=1,470  label=attrition_flag      rate=16.1%
Population B  n=3,000  label=is_voluntary_exit   rate=10.7%


Run `python -m hrai.profiling.relationships` to regenerate `docs/data_relationships.md`.